# مقارنة المحركات بتوقيت mp3quran · Compare the engines with mp3quran timings

يقرأ ملفات النوتبوكين الآخرين من Drive (لا يعيد التوقيت)، ويقارن كل آية بتوقيت mp3quran
(`api/v3/ayat_timing`؛ رقم المصحف يُعرف من رابط الصوت).
Reads the other notebooks' results from Drive and compares every ayah with mp3quran's timings.

**المقاييس · Metrics**
- `start_diff`: متوسط الفرق المطلق عن **بداية** الآية بالثواني (المقياس الأساسي).
- `over_1s_%`: نسبة الآيات التي فرق بدايتها أكثر من ثانية، **والآيات الناقصة تُحسب منها**.
- `end_diff`: فرق النهاية، للاطلاع فقط: نهاية الآية في mp3quran هي بداية التالية (تشمل الصمت)،
  أما المحركان فينهيان الآية عند آخر صوت، فجزء من هذا الفرق ليس خطأ.
- `missing`: آيات لم يوقّتها المحرك. `x_realtime`: كم مرة أسرع من مدة الصوت. `device`: الجهاز الذي شُغّل عليه.
- البسملة (الآية 0) لا تدخل في المقارنة لأن mp3quran لا يوقّتها.

شغّل نوتبوك محرك عثمان ونوتبوك منجّم على العينة أولاً (القيم الافتراضية فيهما هي العينة).

In [ ]:
OUT_DIR = "/content/drive/MyDrive/othman-timings"

In [ ]:
import os
import sys
from google.colab import drive, userdata


def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


drive.mount("/content/drive")
BRANCH = "main"
# GH_TOKEN مطلوب فقط ما دام المستودع خاصاً · only needed while the repository is private
gh = secret("GH_TOKEN")
repo = f"https://{gh}@github.com/vvd7mvv/othman-engine" if gh else "https://github.com/vvd7mvv/othman-engine"
!rm -rf /content/othman-engine && git clone -q --depth 1 -b {BRANCH} {repo} /content/othman-engine
sys.path.insert(0, "/content/othman-engine/notebooks")
sys.modules.pop("timings", None)  # عند إعادة التشغيل في نفس الجلسة: نحمّل النسخة الجديدة لا القديمة من الذاكرة
import timings

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path(OUT_DIR).glob("*/*/*.json")):
    result = json.loads(path.read_text(encoding="utf-8"))
    read = result.get("mp3quran_read")
    if not read:
        continue
    metrics = timings.compare(result, timings.mp3quran_reference(read, result["surah"]))
    meta = result.get("meta", {})
    rows.append({"engine": path.parts[-3], "reciter": path.parts[-2], "surah": result["surah"], **metrics,
                 "seconds": meta.get("seconds"), "audio_seconds": meta.get("audio_seconds"),
                 "device": meta.get("device")})

table = pd.DataFrame(rows)
if table.empty:
    raise SystemExit("لا توجد نتائج في OUT_DIR بعد · No results in OUT_DIR yet")
for engine in ("othman-engine", "munajjam", "munajjam-whisperx"):
    count = (table["engine"] == engine).sum()
    print(f"{'✓' if count else '✗'} {engine}: {count} surahs" + ("" if count else "  ← لم يُشغَّل بعد · not run yet"))
table["x_realtime"] = (table["audio_seconds"] / table["seconds"]).round(1)
table.to_csv(Path(OUT_DIR) / "comparison.csv", index=False)
pd.set_option("display.width", 200)
table.sort_values(["reciter", "surah", "engine"])

## الملخص لكل محرك · Summary per engine
المتوسطات موزونة بعدد الآيات · Means are weighted by the number of ayahs.

In [ ]:
def summarize(group):
    timed = group["ayahs"] - group["missing"]
    return pd.Series({
        "surahs": len(group),
        "ayahs": int(group["ayahs"].sum()),
        "missing": int(group["missing"].sum()),
        "start_diff": round((group["mean_start_diff"] * timed).sum() / timed.sum(), 3),
        "over_1s_%": round((group["over_1s_pct"] * group["ayahs"]).sum() / group["ayahs"].sum(), 1),
        "end_diff": round((group["mean_end_diff"] * timed).sum() / timed.sum(), 3),
        "minutes": round(group["seconds"].sum() / 60, 1),
        "x_realtime": round(group["audio_seconds"].sum() / group["seconds"].sum(), 1),
        "device": ", ".join(sorted(set(group["device"].dropna()))),
    })


summary = table.groupby("engine").apply(summarize, include_groups=False)
summary.to_csv(Path(OUT_DIR) / "comparison_summary.csv")
summary

## حسب القارئ · Per reciter

In [ ]:
table.pivot_table(index=["reciter", "surah"], columns="engine", values=["mean_start_diff", "over_1s_pct", "missing"])